# 中证800 V63：full 模型组合构建消融、板块约束与持仓缓冲

本 notebook 不训练新模型，只使用 V61 导出的 `full` 模型打分面板，测试组合构建是否能改善路径。

核心问题：

1. `top6` 是否过于集中？`top8/top10` 是否能降低回撤而不过度牺牲收益？
2. 创业板/科创板暴露是否需要上限？
3. 老持仓仍在高分区间时继续持有，是否能降低换手和路径噪声？
4. 如果 V62 健康状态可用，能否在 Red/Yellow 时扩大持仓或加强板块约束？

输出是 proxy 组合实验，不替代 JQ-like 日度持仓回测；胜出的规则后续需要并入 V61/V63 的 JQ-like 模拟或真实 JoinQuant 回测验证。

In [ ]:
import os
import ast
import warnings
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings("ignore", category=RuntimeWarning)

V61_OUT_DIR = Path("csi800_ml_v61_jq_like_portfolio_sim_outputs")
V62_OUT_DIR = Path("csi800_ml_v62_mainline_health_alpha_audit_outputs")
OUT_DIR = Path("csi800_ml_v63_portfolio_construction_ablation_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_COL = "rebalance_date"
STOCK_COL = "stock"
SCORE_COL = "score"
RANDOM_N = 500
RANDOM_SEED = 42
SHOW_RANDOM_PROGRESS = True

PRIMARY_MODEL_ID = None  # None: 自动选择 full_2019_2024；也可以手工指定。
print("V61:", V61_OUT_DIR.resolve())
print("V62:", V62_OUT_DIR.resolve())
print("OUT:", OUT_DIR.resolve())


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        print("loaded", path.name, df.shape)
        return df
    print("missing", path.name)
    return pd.DataFrame()

score_panel_df = read_csv_if_exists(V61_OUT_DIR / "v61_score_panel.csv")
model_summary_df = read_csv_if_exists(V61_OUT_DIR / "v61_model_summary.csv")
health_state_df = read_csv_if_exists(V62_OUT_DIR / "v62_health_state.csv")

if score_panel_df.empty:
    raise ValueError("缺少 v61_score_panel.csv，请先运行 full-only V61。")

score_panel_df[DATE_COL] = pd.to_datetime(score_panel_df[DATE_COL])
if not health_state_df.empty and DATE_COL in health_state_df.columns:
    health_state_df[DATE_COL] = pd.to_datetime(health_state_df[DATE_COL])

In [ ]:
def first_existing_col(df: pd.DataFrame, cols: List[str]) -> Optional[str]:
    for c in cols:
        if c in df.columns:
            return c
    return None


def infer_future_return_col(df: pd.DataFrame) -> Optional[str]:
    candidates = ["alpha_1m", "realized_raw_ret", "raw_return_1m", "stock_return_1m", "future_return_1m", "return_1m", "label", "y"]
    for c in candidates:
        if c in df.columns and pd.to_numeric(df[c], errors="coerce").notnull().any():
            return c
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in ["alpha", "return", "future", "ret_1m"]):
            if pd.to_numeric(df[c], errors="coerce").notnull().any():
                return c
    return None

future_col = infer_future_return_col(score_panel_df)
raw_col = first_existing_col(score_panel_df, ["raw_return_1m", "stock_return_1m", "realized_raw_ret", "future_raw_return_1m"])
if raw_col is None:
    raw_col = future_col
print("future_col:", future_col, "raw_col:", raw_col)
if future_col is None:
    raise ValueError("score panel 中没有可用未来收益/alpha 列。")


def choose_primary_model(score_df: pd.DataFrame, summary_df: pd.DataFrame, requested: Optional[str]) -> str:
    if requested is not None:
        return str(requested)
    ids = sorted(score_df["model_id"].astype(str).unique())
    preferred = [x for x in ids if "full_2019_2024" in x]
    if preferred:
        return preferred[0]
    preferred = [x for x in ids if "full" in x]
    if preferred:
        return preferred[0]
    if not summary_df.empty and "model_id" in summary_df.columns:
        return str(summary_df.iloc[0]["model_id"])
    return ids[0]

PRIMARY_MODEL_ID = choose_primary_model(score_panel_df, model_summary_df, PRIMARY_MODEL_ID)
print("PRIMARY_MODEL_ID:", PRIMARY_MODEL_ID)
score_panel_df = score_panel_df[score_panel_df["model_id"].astype(str) == str(PRIMARY_MODEL_ID)].copy()

In [ ]:
def board_from_stock(stock: str) -> str:
    s = str(stock)
    if s.startswith("688"):
        return "STAR"
    if s.startswith("30"):
        return "ChiNext"
    if s.startswith(("8", "4", "43", "87", "92")):
        return "BSE"
    if s.startswith("6"):
        return "SH_main"
    if s.startswith(("0", "2")):
        return "SZ_main"
    return "unknown"


def board_ok(selected: List[str], stock: str, board_caps: Optional[Dict[str, int]]) -> bool:
    if not board_caps:
        return True
    b = board_from_stock(stock)
    if b not in board_caps:
        return True
    current = sum(1 for x in selected if board_from_stock(x) == b)
    return current < int(board_caps[b])


def select_targets(month_df: pd.DataFrame, top_n: int, board_caps: Optional[Dict[str, int]] = None,
                   prev_targets: Optional[List[str]] = None, keep_rank: Optional[int] = None,
                   random_score_col: Optional[str] = None) -> List[str]:
    score_col = random_score_col if random_score_col is not None else SCORE_COL
    m = month_df.dropna(subset=[score_col]).copy().sort_values(score_col, ascending=False)
    stocks = list(m[STOCK_COL].astype(str))
    rank_map = {s: i + 1 for i, s in enumerate(stocks)}
    selected = []
    if prev_targets and keep_rank:
        for s in prev_targets:
            if s in rank_map and rank_map[s] <= keep_rank and board_ok(selected, s, board_caps):
                selected.append(s)
                if len(selected) >= top_n:
                    return selected
    for s in stocks:
        if s in selected:
            continue
        if board_ok(selected, s, board_caps):
            selected.append(s)
        if len(selected) >= top_n:
            break
    if len(selected) < top_n:
        for s in stocks:
            if s not in selected:
                selected.append(s)
            if len(selected) >= top_n:
                break
    return selected[:top_n]


def mean_ret(month_df: pd.DataFrame, targets: List[str], col: str) -> float:
    vals = pd.to_numeric(month_df[month_df[STOCK_COL].astype(str).isin(targets)][col], errors="coerce").dropna()
    return float(vals.mean()) if len(vals) else np.nan


def turnover(prev: List[str], curr: List[str]) -> float:
    if not curr:
        return np.nan
    if not prev:
        return 1.0
    overlap = len(set(prev) & set(curr))
    return 1.0 - overlap / float(max(len(curr), 1))


def max_drawdown(rets: pd.Series) -> float:
    r = pd.to_numeric(rets, errors="coerce").fillna(0)
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1
    return float(dd.min()) if len(dd) else np.nan


def cum_ret(rets: pd.Series) -> float:
    r = pd.to_numeric(rets, errors="coerce").dropna()
    return float((1 + r).prod() - 1) if len(r) else np.nan


def ann_ret(rets: pd.Series) -> float:
    r = pd.to_numeric(rets, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    total = (1 + r).prod() - 1
    years = len(r) / 12.0
    if years <= 0 or total <= -1:
        return np.nan
    return float((1 + total) ** (1 / years) - 1)

In [ ]:
# 组合实验清单。board_caps 是绝对持仓数量上限，不是比例。
PORTFOLIO_SPECS = [
    {"name": "top6_baseline", "top_n": 6, "board_caps": None, "keep_rank": None},
    {"name": "top8", "top_n": 8, "board_caps": None, "keep_rank": None},
    {"name": "top10", "top_n": 10, "board_caps": None, "keep_rank": None},
    {"name": "top6_board_cap", "top_n": 6, "board_caps": {"ChiNext": 2, "STAR": 1}, "keep_rank": None},
    {"name": "top8_board_cap", "top_n": 8, "board_caps": {"ChiNext": 3, "STAR": 2}, "keep_rank": None},
    {"name": "top10_board_cap", "top_n": 10, "board_caps": {"ChiNext": 4, "STAR": 2}, "keep_rank": None},
    {"name": "top6_keep30", "top_n": 6, "board_caps": None, "keep_rank": 30},
    {"name": "top8_keep30_board_cap", "top_n": 8, "board_caps": {"ChiNext": 3, "STAR": 2}, "keep_rank": 30},
    {"name": "health_adaptive", "top_n": 8, "board_caps": {"ChiNext": 3, "STAR": 2}, "keep_rank": 30, "health_adaptive": True},
]

health_map = {}
if not health_state_df.empty and "health_state" in health_state_df.columns:
    tmp = health_state_df[health_state_df["model_id"].astype(str) == str(PRIMARY_MODEL_ID)].copy() if "model_id" in health_state_df.columns else health_state_df.copy()
    health_map = dict(zip(pd.to_datetime(tmp[DATE_COL]), tmp["health_state"].astype(str)))
print("health states available:", len(health_map))

In [ ]:
def effective_spec(spec: Dict[str, Any], dt: pd.Timestamp) -> Dict[str, Any]:
    out = dict(spec)
    if spec.get("health_adaptive"):
        state = health_map.get(pd.Timestamp(dt), "Yellow")
        out["health_state"] = state
        if state == "Green":
            out["top_n"] = 6
            out["board_caps"] = {"ChiNext": 2, "STAR": 1}
        elif state == "Red":
            out["top_n"] = 10
            out["board_caps"] = {"ChiNext": 4, "STAR": 2}
        else:
            out["top_n"] = 8
            out["board_caps"] = {"ChiNext": 3, "STAR": 2}
    return out


def random_percentile_for_spec(month_df: pd.DataFrame, actual_targets: List[str], spec_eff: Dict[str, Any], actual_ret: float, dt: pd.Timestamp) -> float:
    if not actual_targets or pd.isna(actual_ret):
        return np.nan
    rng = np.random.RandomState(RANDOM_SEED + int(pd.Timestamp(dt).strftime("%Y%m%d")) % 100000)
    vals = []
    tmp = month_df.copy().reset_index(drop=True)
    rand_iter = range(RANDOM_N)
    if SHOW_RANDOM_PROGRESS:
        rand_iter = tqdm(rand_iter, desc="random %s %s" % (spec_eff.get("name", "spec"), pd.Timestamp(dt).strftime("%Y-%m")), leave=False)
    for _ in rand_iter:
        tmp["_rand_score"] = rng.normal(size=len(tmp))
        rt = select_targets(tmp, spec_eff["top_n"], spec_eff.get("board_caps"), prev_targets=None, keep_rank=None, random_score_col="_rand_score")
        vals.append(mean_ret(tmp, rt, future_col))
    vals = pd.Series(vals).dropna()
    return float((vals <= actual_ret).mean()) if len(vals) else np.nan


def run_one_spec(score_df: pd.DataFrame, spec: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    prev_targets = []
    grouped = list(score_df.sort_values(DATE_COL).groupby(DATE_COL))
    month_iter = tqdm(grouped, desc="months %s" % spec.get("name", "spec"), leave=False)
    for dt, month_df in month_iter:
        m = month_df.dropna(subset=[SCORE_COL, future_col]).copy()
        spec_eff = effective_spec(spec, dt)
        targets = select_targets(m, spec_eff["top_n"], spec_eff.get("board_caps"), prev_targets=prev_targets, keep_rank=spec_eff.get("keep_rank"))
        proxy_excess = mean_ret(m, targets, future_col)
        proxy_raw = mean_ret(m, targets, raw_col) if raw_col is not None else proxy_excess
        rand_pct = random_percentile_for_spec(m, targets, spec_eff, proxy_excess, dt)
        real_sorted = m.sort_values(future_col, ascending=False)
        top20 = set(real_sorted.head(20)[STOCK_COL].astype(str))
        top10 = set(real_sorted.head(10)[STOCK_COL].astype(str))
        boards = [board_from_stock(x) for x in targets]
        rows.append({
            "spec": spec["name"],
            DATE_COL: dt,
            "model_id": PRIMARY_MODEL_ID,
            "health_state": spec_eff.get("health_state", health_map.get(pd.Timestamp(dt), "")),
            "top_n": spec_eff["top_n"],
            "board_caps": str(spec_eff.get("board_caps")),
            "keep_rank": spec_eff.get("keep_rank"),
            "targets": str(targets),
            "proxy_excess_ret": proxy_excess,
            "proxy_raw_ret": proxy_raw,
            "turnover": turnover(prev_targets, targets),
            "random_percentile": rand_pct,
            "selected_hit_real_top10": len(set(targets) & top10),
            "selected_hit_real_top20": len(set(targets) & top20),
            "selected_hit_rate_real_top20": len(set(targets) & top20) / max(len(targets), 1),
            "main_board_ratio": sum(b in ("SH_main", "SZ_main") for b in boards) / max(len(boards), 1),
            "chinext_ratio": boards.count("ChiNext") / max(len(boards), 1),
            "star_ratio": boards.count("STAR") / max(len(boards), 1),
            "board_hhi": sum((boards.count(b) / max(len(boards), 1)) ** 2 for b in set(boards)) if boards else np.nan,
        })
        prev_targets = targets
    return pd.DataFrame(rows)

monthly_parts = []
for spec in tqdm(PORTFOLIO_SPECS, desc="portfolio specs"):
    monthly_parts.append(run_one_spec(score_panel_df, spec))
v63_monthly_df = pd.concat(monthly_parts, ignore_index=True)
display(v63_monthly_df.tail(20))


In [ ]:
def summarize_spec(g: pd.DataFrame) -> Dict[str, Any]:
    return {
        "spec": g["spec"].iloc[0],
        "months": int(len(g)),
        "proxy_excess_cum_ret": cum_ret(g["proxy_excess_ret"]),
        "proxy_excess_ann_ret": ann_ret(g["proxy_excess_ret"]),
        "proxy_excess_max_drawdown": max_drawdown(g["proxy_excess_ret"]),
        "proxy_raw_cum_ret": cum_ret(g["proxy_raw_ret"]),
        "proxy_raw_ann_ret": ann_ret(g["proxy_raw_ret"]),
        "proxy_raw_max_drawdown": max_drawdown(g["proxy_raw_ret"]),
        "avg_turnover": float(pd.to_numeric(g["turnover"], errors="coerce").mean()),
        "avg_random_percentile": float(pd.to_numeric(g["random_percentile"], errors="coerce").mean()),
        "avg_selected_hit_rate_real_top20": float(pd.to_numeric(g["selected_hit_rate_real_top20"], errors="coerce").mean()),
        "avg_board_hhi": float(pd.to_numeric(g["board_hhi"], errors="coerce").mean()),
        "avg_chinext_ratio": float(pd.to_numeric(g["chinext_ratio"], errors="coerce").mean()),
        "avg_star_ratio": float(pd.to_numeric(g["star_ratio"], errors="coerce").mean()),
    }

summary_groups = list(v63_monthly_df.groupby("spec"))
v63_summary_df = pd.DataFrame([summarize_spec(g) for _, g in tqdm(summary_groups, desc="summarize specs")])
v63_summary_df = v63_summary_df.sort_values(["proxy_excess_cum_ret", "proxy_excess_max_drawdown"], ascending=[False, False])
display(v63_summary_df)

latest_dt = v63_monthly_df[DATE_COL].max()
v63_latest_targets_df = v63_monthly_df[v63_monthly_df[DATE_COL] == latest_dt].copy()
display(v63_latest_targets_df[["spec", DATE_COL, "top_n", "targets", "proxy_excess_ret", "random_percentile", "board_hhi", "chinext_ratio", "star_ratio"]])


In [ ]:
v63_monthly_df.to_csv(OUT_DIR / "v63_portfolio_ablation_monthly.csv", index=False)
v63_summary_df.to_csv(OUT_DIR / "v63_portfolio_ablation_summary.csv", index=False)
v63_latest_targets_df.to_csv(OUT_DIR / "v63_latest_targets_by_spec.csv", index=False)
print("saved:")
for p in sorted(OUT_DIR.glob("v63_*.csv")):
    print("-", p)

## 读数规则

优先看：

1. `v63_portfolio_ablation_summary.csv`：组合规则整体排序。
2. `proxy_excess_cum_ret` 和 `proxy_excess_max_drawdown`：收益和回撤必须一起看。
3. `avg_turnover`：持仓缓冲是否真的降低换手。
4. `avg_random_percentile`：组合是否强于同月同板块约束随机。
5. `avg_board_hhi / avg_chinext_ratio / avg_star_ratio`：收益是否来自过度板块集中。

保留规则：

- 如果某个组合规则收益略低但回撤、换手、板块集中明显更好，可以进入 JQ-like 复测。
- 如果某个规则只提高 proxy alpha，但 random percentile 不高，或板块暴露极端，不直接进入主线。
- V63 是组合层 proxy 实验；最终仍需在 V61/JQ-like 或真实 JoinQuant 回测中复核。